## Task 3.1 — Tạo Ground Truth 5 trạng thái (có Murmur)

### Mục tiêu

Chuyển nhãn phân đoạn 4 trạng thái gốc từ file `.tsv` (S1, Systole, S2, Diastole) thành nhãn 5 trạng thái cấp frame bằng cách ghi nhãn lại một phần Systole thành Murmur, dựa trên chú thích thời điểm tiếng thổi của bác sĩ.

### Tại sao quan trọng

File `.tsv` gốc chỉ có 4 trạng thái — không biết đâu là tiếng thổi. Đổi mới chính của bài báo là dùng metadata timing (Early-systolic, Mid-systolic, Holosystolic) để xấp xỉ vị trí murmur trong mỗi đoạn tâm thu. Nếu không có bước này, RNN không thể học phát hiện tiếng thổi.

### Logic ghi nhãn lại (từ bài báo)

Ánh xạ trạng thái: TSV 1→S1(0), 2→Systole(1), 3→S2(2), 4→Diastole(3), TSV 0→Unannotated(-1). Trạng thái mới: Murmur(4).

Với mỗi đoạn Systole trong bản ghi có tiếng thổi tại vị trí nghe đó:

- Holosystolic: toàn bộ systole → Murmur
- Early-systolic: 50% đầu → Murmur, 50% sau giữ Systole
- Mid-systolic: 50% giữa → Murmur, 25% đầu + 25% cuối giữ Systole
- Late-systolic: 50% cuối → Murmur (chỉ 1 case)
- Diastolic: KHÔNG ghi nhãn lại (bài báo nói rõ không mô hình diastolic murmur)

In [58]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Thêm project root vào Python path
project_root = Path.cwd().parent  # notebooks/ → lên 1 cấp
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"src exists: {(project_root / 'src').exists()}")

Project root: D:\1 Onedrive\1 Lectures\3.2 DAV\papers\heart-murmur-detection
src exists: True


In [59]:
DATA_ROOT   = project_root / "data" / "raw" / "training_data"
SPEC_DIR    = project_root / "data" / "processed" / "spectrograms"
LABEL_DIR   = project_root / "data" / "processed" / "labels"
META_DIR    = project_root / "data" / "metadata"

SPEC_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT  exists: {DATA_ROOT.exists()}")
print(f"SPEC_DIR   exists: {SPEC_DIR.exists()}")
print(f"LABEL_DIR  exists: {LABEL_DIR.exists()}")

DATA_ROOT  exists: True
SPEC_DIR   exists: True
LABEL_DIR  exists: True


In [60]:
# Thêm cột recording_id đúng vào recordings_df
recordings_df['recording_id'] = recordings_df['wav_path'].apply(
    lambda p: Path(p).stem
)

# Kiểm tra trước khi lưu
n_dupes = recordings_df['recording_id'].duplicated().sum()
print(f"recording_id bị trùng: {n_dupes}")  # phải = 0
print(f"Total rows: {len(recordings_df)}")   # phải = 3163
print(recordings_df[['patient_id', 'location', 'recording_id']].head(5))

# Lưu lại CSV
recordings_df.to_csv(META_DIR / "recordings.csv", index=False)
print("\nĐã lưu recordings.csv với cột recording_id")

recording_id bị trùng: 0
Total rows: 3163
   patient_id location recording_id
0        2530       AV      2530_AV
1        2530       PV      2530_PV
2        2530       TV      2530_TV
3        2530       MV      2530_MV
4        9979       AV      9979_AV

Đã lưu recordings.csv với cột recording_id


In [61]:
## Extract spectrograms
from src.features.spectrogram import extract_features

failed = []

for _, row in tqdm(recordings_df.iterrows(), total=len(recordings_df),
                   desc="Extracting spectrograms"):
    rec_id   = row['recording_id']
    out_path = SPEC_DIR / f"{rec_id}.npy"

    if out_path.exists():
        continue

    wav_path = DATA_ROOT / Path(row['wav_path']).name

    try:
        features, freqs, times = extract_features(wav_path)
        np.save(out_path, features)  # shape (41, T)
    except Exception as e:
        failed.append((rec_id, str(e)))

n_saved = len(list(SPEC_DIR.glob("*.npy")))
print(f"\nSaved : {n_saved} spectrograms")
print(f"Failed: {len(failed)}")
if failed:
    print("First 5 failures:")
    for rec_id, err in failed[:5]:
        print(f"  {rec_id}: {err}")

Extracting spectrograms: 100%|██████████| 3163/3163 [00:15<00:00, 197.84it/s]


Saved : 3163 spectrograms
Failed: 0


In [17]:
## tạo 5 nhãn trạng thái => data/processed/labels/ — 3163 file .npy shape (T,), dtype int8
from src.data.labels import create_all_labels

stats = create_all_labels(
    recordings_df  = recordings_df,
    patients_df    = patients_df,
    data_root      = DATA_ROOT,
    spectrogram_dir= SPEC_DIR,
    output_dir     = LABEL_DIR,
    feature_rate   = 50,
    verbose        = True
)

  [500/3163] Processing labels...
  [1000/3163] Processing labels...
  [1500/3163] Processing labels...
  [2000/3163] Processing labels...
  [2500/3163] Processing labels...
  [3000/3163] Processing labels...

  Done: 3163 processed, 0 skipped (no spectrogram), 0 skipped (no TSV), 0 errors
  Recordings with Murmur frames: 497

  Frame counts:
              S1:    385,582  ( 20.3% of labelled)
         Systole:    378,660  ( 19.9% of labelled)
              S2:    337,989  ( 17.8% of labelled)
        Diastole:    741,055  ( 39.0% of labelled)
          Murmur:     55,591  (  2.9% of labelled)
     Unannotated:  1,722,462

  Murmur timing distribution:
          Early-systolic: 145
            Holosystolic: 294
           Late-systolic: 2
            Mid-systolic: 56


In [16]:
from src.data.labels import inspect_recording_labels

for rec_id in ["2530_MV", "9979_TV", "9983_MV"]:
    inspect_recording_labels(rec_id, LABEL_DIR)


  Recording: 2530_MV
  T = 1336 frames (26.7 seconds)
  Label distribution:
     Unannotated:    134 frames ( 10.0%)
              S1:    261 frames ( 19.5%)
         Systole:    280 frames ( 21.0%)
              S2:    273 frames ( 20.4%)
        Diastole:    388 frames ( 29.0%)
          Murmur:      0 frames (  0.0%)
  Contains Murmur: False
  Number of segments: 175
  First 20 segments:
    [    0–   50]  Unannotated  (1000 ms)
    [   50–   56]           S1  (120 ms)
    [   56–   65]      Systole  (180 ms)
    [   65–   72]           S2  (140 ms)
    [   72–   84]     Diastole  (240 ms)
    [   84–   89]           S1  (100 ms)
    [   89–   97]      Systole  (160 ms)
    [   97–  103]           S2  (120 ms)
    [  103–  119]     Diastole  (320 ms)
    [  119–  125]           S2  (120 ms)
    [  125–  133]     Diastole  (160 ms)
    [  133–  138]           S1  (100 ms)
    [  138–  152]     Diastole  (280 ms)
    [  152–  159]           S1  (140 ms)
    [  159–  167]      Systole

## ✅ Task 3.1 — Kết quả

### Những gì đã làm
1. **Fix `recording_id`** — phát hiện 22 bản ghi bị trùng do ghép tay `patient_id + location`.
   Sửa bằng cách lấy stem từ `wav_path` (ví dụ `49748_AV_1.wav` → `49748_AV_1`).
   Lưu lại vào `recordings.csv` — không cần tạo lại cột này ở các phase sau.

2. **Extract spectrograms** — chạy `extract_features()` (Phase 2 module) trên toàn bộ dataset.
   Lưu 3163 file `.npy` shape `(41, T)` vào `data/processed/spectrograms/`.

3. **Tạo nhãn 5 trạng thái** — `src/data/labels.py` chuyển đổi file `.tsv` gốc (4 trạng thái)
   thành nhãn cấp frame `(T,)` với 5 trạng thái: S1=0, Systole=1, S2=2, Diastole=3, Murmur=4,
   Unannotated=-1.

   Logic ghi nhãn lại Murmur (chỉ áp dụng nếu bản ghi thuộc bệnh nhân Present **và**
   vị trí nghe nằm trong `murmur_locations`):
   - **Holosystolic** → toàn bộ systole thành Murmur
   - **Early-systolic** → 50% đầu mỗi đoạn systole thành Murmur
   - **Mid-systolic** → 50% giữa mỗi đoạn systole thành Murmur
   - **Late-systolic** → 50% cuối mỗi đoạn systole thành Murmur
   - **Diastolic** → không ghi nhãn lại (bài báo không mô hình diastolic murmur)

### Kết quả kiểm chứng

| Chỉ số | Giá trị | Nhận xét |
|--------|---------|----------|
| Recordings đã xử lý | 3163 / 3163 | Không có lỗi |
| Recordings có Murmur frame | 497 | ~179 bệnh nhân × 2.8 vị trí trung bình ✓ |
| Murmur % tổng frame có nhãn | 2.9% | Hiếm — cần class-weighted loss ✓ |
| Diastole % | 39.0% | Pha dài nhất, chiếm ưu thế ✓ |
| Timing: Holosystolic / Early / Mid / Late | 294 / 145 / 56 / 2 | Khớp Phase 1 EDA (57%/33%/10%/0.6%) ✓ |

### Kiểm tra 3 bản ghi mẫu

| Recording | Label | Murmur frames | Kỳ vọng | Kết quả |
|-----------|-------|:---:|---------|---------|
| `2530_MV` | Absent | 0 | Không có murmur | ✅ |
| `9979_TV` | Present, Holosystolic | 52 | Toàn systole → murmur (Systole=0) | ✅ |
| `9983_MV` | Unknown | 0 | Không có murmur, annotation thưa | ✅ |

### Artifacts tạo ra
- `data/processed/spectrograms/` — 3163 file `.npy` shape `(41, T)`
- `data/processed/labels/` — 3163 file `.npy` shape `(T,)`, dtype int8
- `data/metadata/recordings.csv` — cập nhật thêm cột `recording_id`
- `src/data/labels.py` — module tái sử dụng được

### Điểm quan trọng cho các task tiếp theo
- Frame Unannotated (`-1`) chiếm **47%** tổng frame — phải dùng `ignore_index=-1`
  trong CrossEntropyLoss, không được coi là lớp hợp lệ.
- Murmur chỉ 2.9% frame có nhãn → **class-weighted loss là bắt buộc** (Task 3.4).
- `recording_id` cho bản ghi multi-location có dạng `49748_AV_1`, `49748_AV_2` —
  phải dùng cột này thay vì ghép tay từ `patient_id + location`.

## Task 3.2 — Tải dữ liệu và batching cho chuỗi độ dài thay đổi

### Đang làm gì
Xây dựng PyTorch `Dataset` và `DataLoader` để tải spectrogram + nhãn đã tính
trước (Task 3.1) và đóng gói thành batch cho việc huấn luyện RNN. (them code loader.py)

### Vấn đề cần giải quyết
Mỗi bản ghi PCG có thời lượng khác nhau → T dao động từ ~250 đến ~3250 frame.
PyTorch yêu cầu tất cả tensor trong cùng một batch phải có shape giống nhau.

### Cách làm
1. **`PCGDataset`** — tải file `.npy` từ disk, transpose spectrogram từ `(41, T)`
   sang `(T, 41)` (RNN cần time là chiều đầu tiên).
2. **`pcg_collate_fn`** — gom các sample trong batch:
   - Sắp xếp theo độ dài giảm dần (bắt buộc cho `pack_padded_sequence`)
   - Pad features bằng `0.0`, pad labels bằng `-1`
3. **`create_dataloader`** — wrapper tạo DataLoader với collate_fn trên.

### Tại sao pad labels bằng -1
`CrossEntropyLoss(ignore_index=-1)` sẽ bỏ qua tất cả vị trí có nhãn `-1`
khi tính loss — gồm cả frame Unannotated (từ Task 3.1) lẫn frame padding.
Hai loại frame này đều không nên đóng góp vào gradient.

### Kết quả mong muốn
- Mỗi batch trả về: `features (B, T_max, 41)`, `labels (B, T_max)`, `lengths [B]`
- `lengths` giảm dần trong mỗi batch
- Vùng padding có `labels = -1`
- DataLoader chạy qua toàn bộ 3163 bản ghi không lỗi
- Module lưu tại `src/data/loader.py` — tái sử dụng xuyên suốt Phase 3
```

In [32]:
## Import và kiểm tra Dataset

from src.data.loader import PCGDataset, pcg_collate_fn, create_dataloader

# Lấy 5 recording_id đầu tiên để test
test_ids = recordings_df['recording_id'].tolist()[:5]

dataset = PCGDataset(test_ids, SPEC_DIR, LABEL_DIR)
print(f"Dataset size: {len(dataset)}")

# Kiểm tra 1 item
item = dataset[0]
print(f"\nItem 0 — {item['recording_id']}:")
print(f"  features shape : {item['features'].shape}")   # kỳ vọng (T, 41)
print(f"  labels shape   : {item['labels'].shape}")     # kỳ vọng (T,)
print(f"  length         : {item['length']}")           # kỳ vọng = T
print(f"  features dtype : {item['features'].dtype}")   # kỳ vọng torch.float32
print(f"  labels dtype   : {item['labels'].dtype}")     # kỳ vọng torch.int64
print(f"  unique labels  : {item['labels'].unique()}")  # kỳ vọng subset of {-1,0,1,2,3,4}

Dataset size: 5

Item 0 — 2530_AV:
  features shape : torch.Size([1181, 41])
  labels shape   : torch.Size([1181])
  length         : 1181
  features dtype : torch.float32
  labels dtype   : torch.int64
  unique labels  : tensor([-1,  0,  1,  2,  3])


In [34]:
##Kiểm tra collate_fn (batch)
from torch.utils.data import DataLoader

# Tạo loader nhỏ không shuffle để kết quả tái tạo được
loader = create_dataloader(test_ids, SPEC_DIR, LABEL_DIR,
                           batch_size=5, shuffle=False)

batch = next(iter(loader))

print("=== Batch shapes ===")
print(f"features : {batch['features'].shape}")   # (5, T_max, 41)
print(f"labels   : {batch['labels'].shape}")     # (5, T_max)
print(f"lengths  : {batch['lengths']}")          # giảm dần

print("\n=== Kiểm tra padding ===")
for i, (rec_id, T) in enumerate(zip(batch['recording_ids'], batch['lengths'])):
    # Vùng padding phải là -1
    if T < batch['labels'].shape[1]:
        pad_labels = batch['labels'][i, T:].unique().tolist()
        print(f"  {rec_id} (T={T}): padding labels = {pad_labels}")  # phải = [-1]
    else:
        print(f"  {rec_id} (T={T}): longest sequence, no padding")

print("\n=== Kiểm tra thứ tự sắp xếp ===")
lengths = batch['lengths']
is_sorted = all(lengths[i] >= lengths[i+1] for i in range(len(lengths)-1))
print(f"Lengths sorted descending: {is_sorted}")  # phải True

=== Batch shapes ===
features : torch.Size([5, 1336, 41])
labels   : torch.Size([5, 1336])
lengths  : [1336, 1181, 909, 751, 654]

=== Kiểm tra padding ===
  2530_MV (T=1336): longest sequence, no padding
  2530_AV (T=1181): padding labels = [-1]
  9979_AV (T=909): padding labels = [-1]
  2530_TV (T=751): padding labels = [-1]
  2530_PV (T=654): padding labels = [-1]

=== Kiểm tra thứ tự sắp xếp ===
Lengths sorted descending: True


features: torch.Size([5, 1336, 41])

Batch có 5 bản ghi, tất cả được pad đến T_max=1336 frame (bản ghi dài nhất), mỗi frame có 41 frequency bin. Đây đúng là shape (B, T_max, 41) mà GRU cần nhận.
lengths: [1336, 1181, 909, 751, 654]

Độ dài thực tế của 5 bản ghi, giảm dần — bắt buộc cho pack_padded_sequence. Khi ta truyền list này vào GRU, PyTorch biết bản ghi thứ 2 chỉ có 1181 frame thực sự, không tính 155 frame padding cuối.
2530_MV (T=1336): longest sequence, no padding

Bản ghi dài nhất không cần padding — đây là "chuẩn" mà các bản ghi ngắn hơn pad vào.
2530_AV (T=1181): padding labels = [-1]

155 frame cuối của bản ghi này (từ vị trí 1181 đến 1335) được gán nhãn -1. Khi CrossEntropyLoss(ignore_index=-1) gặp các frame này, nó bỏ qua hoàn toàn — gradient không chảy từ padding, mô hình không học gì từ các vị trí giả tạo này.
Lengths sorted descending: True

Xác nhận thứ tự sắp xếp đúng.

In [30]:
##Kiểm tra toàn bộ dataset 
all_ids = recordings_df['recording_id'].tolist()
full_loader = create_dataloader(all_ids, SPEC_DIR, LABEL_DIR,
                                batch_size=32, shuffle=False)

print(f"Total batches: {len(full_loader)}")
print(f"Total recordings: {len(full_loader.dataset)}")

# Chạy qua 1 epoch để xác nhận không có lỗi
n_frames_total = 0
for batch in full_loader:
    n_frames_total += sum(batch['lengths'])

print(f"Total frames processed: {n_frames_total:,}")
print("✅ No errors — DataLoader works on full dataset")

Total batches: 99
Total recordings: 3163
Total frames processed: 3,621,339
✅ No errors — DataLoader works on full dataset


## ✅ Task 3.2 — Kết quả

### Artifacts tạo ra
- `src/data/loader.py` — `PCGDataset`, `pcg_collate_fn`, `create_dataloader`

### Kiểm chứng

| Kiểm tra | Kết quả | Nhận xét |
|----------|---------|----------|
| Item shape | `features (T, 41)`, `labels (T,)` | Đúng chiều cho RNN ✅ |
| Batch shape | `(5, 1336, 41)` | T_max = bản ghi dài nhất trong batch ✅ |
| Lengths sorted | `[1336, 1181, 909, 751, 654]` giảm dần | Đúng yêu cầu `pack_padded_sequence` ✅ |
| Padding labels | `-1` tại vùng padding | Sẽ bị `ignore_index=-1` bỏ qua trong loss ✅ |
| Full dataset | 3163 recordings, 99 batches (batch_size=32) | Không lỗi ✅ |
| Tổng frames | 3,621,339 | Hợp lý (~1145 frame/bản ghi trung bình) ✅ |

## Task 3.3 — Triển khai kiến trúc Bidirectional GRU + FC Head

### Đang làm gì
Xây dựng mạng nơ-ron `MurmurRNN` — trái tim của pipeline phát hiện tiếng thổi.
Mạng nhận spectrogram `(T, 41)` và xuất xác suất hậu nghiệm 5 trạng thái `(T, 5)`
cho mỗi frame thời gian.

### Kiến trúc (từ bài báo)

```text
Input x₁:T  (41 features mỗi frame)
        ↓
[3-layer Bidirectional GRU, hidden=60]
    → Forward GRU:   đọc từ t=1 → T
    → Backward GRU:  đọc từ t=T → 1
    → Concat output: dim = 60×2 = 120 tại mỗi time-step
    
        ↓ Dropout (p=0.1)
        
[FC layer: 120 → 60, activation Tanh]
        ↓
Dropout (p=0.1)
        ↓
[FC layer: 60 → 40, activation Tanh]
        ↓
[FC layer: 40 → 5]
        ↓
[Softmax]
    → P(qₜ = S1 | x₁:T, θ)
    → P(qₜ = systole | x₁:T, θ)
    → P(qₜ = S2 | x₁:T, θ)
    → P(qₜ = diastole | x₁:T, θ)
    → P(qₜ = murmur | x₁:T, θ)
```

### Tại sao không thêm Softmax ở cuối
`nn.CrossEntropyLoss` của PyTorch đã tích hợp log-softmax bên trong.
Nếu thêm softmax trước thì loss sẽ tính sai (softmax của softmax).
Chỉ áp dụng `torch.softmax()` khi inference để lấy xác suất hậu nghiệm.

### Tại sao dùng BiGRU thay vì GRU một chiều
Chiều xuôi biết quá khứ, chiều ngược biết tương lai. Với phân đoạn tim,
biết cái gì đến SAU một âm thanh giúp nhận diện nó chính xác hơn
(ví dụ: diastole luôn theo sau S2 — ngữ cảnh tương lai rất quan trọng).

### Kết quả mong muốn
- Forward pass trên batch giả không lỗi, output shape `(B, T_max, 5)`
- Số tham số ~200k–300k (mô hình nhỏ, tránh overfitting)
- Có thể lưu và tải lại bằng `torch.save` / `torch.load`

In [35]:
33  Kiểm tra kiến trúc và số tham số
import torch
from src.models.rnn import MurmurRNN, count_parameters, build_model

model = build_model(seed=42)
print(model)

total, trainable = count_parameters(model)
print(f"\nTổng tham số    : {total:,}")
print(f"Tham số học được: {trainable:,}")

MurmurRNN(
  (gru): GRU(41, 60, num_layers=3, batch_first=True, dropout=0.1, bidirectional=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (fc1): Linear(in_features=120, out_features=60, bias=True)
  (act1): Tanh()
  (fc2): Linear(in_features=60, out_features=40, bias=True)
  (act2): Tanh()
  (output_layer): Linear(in_features=40, out_features=5, bias=True)
)

Tổng tham số    : 178,025
Tham số học được: 178,025


In [36]:
## Kiểm tra forward pass
torch.manual_seed(42)

# Tạo batch giả: 4 bản ghi, độ dài giảm dần
B, T_max, F = 4, 500, 41
features_fake = torch.randn(B, T_max, F)
lengths_fake  = [500, 420, 310, 200]  # giảm dần

# Zero-pad các bản ghi ngắn hơn
for i, T in enumerate(lengths_fake):
    features_fake[i, T:, :] = 0.0

# Forward pass
model.eval()
with torch.no_grad():
    logits = model(features_fake, lengths_fake)

print(f"Input  shape: {features_fake.shape}")   # (4, 500, 41)
print(f"Output shape: {logits.shape}")           # (4, 500, 5)

# Kiểm tra softmax tổng = 1.0
probs = torch.softmax(logits, dim=-1)
row_sums = probs[0, :lengths_fake[0], :].sum(dim=-1)
print(f"\nProbs sum tại mỗi frame (sample 5 frame đầu):")
print(row_sums[:5])  # phải xấp xỉ [1.0, 1.0, 1.0, 1.0, 1.0]

print(f"\n✅ Forward pass OK — output shape: {logits.shape}")

Input  shape: torch.Size([4, 500, 41])
Output shape: torch.Size([4, 500, 5])

Probs sum tại mỗi frame (sample 5 frame đầu):
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

✅ Forward pass OK — output shape: torch.Size([4, 500, 5])


#### ✅ Task 3.3 — Kết quả

##### Artifacts tạo ra
- `src/models/rnn.py` — `MurmurRNN`, `build_model()`, `count_parameters()`

##### Kiểm chứng

| Kiểm tra | Kết quả | Nhận xét |
|----------|---------|----------|
| Số tham số | 178,025 | Nhỏ gọn, tránh overfitting trên dataset 942 bệnh nhân ✅ |
| Input shape | `(4, 500, 41)` | B × T_max × features ✅ |
| Output shape | `(4, 500, 5)` | B × T_max × num_classes ✅ |
| Probs sum | 1.0000 mỗi frame | Softmax hợp lệ ✅ |

##### Lưu ý quan trọng
- Model xuất **logits thô** (không softmax) → dùng với `CrossEntropyLoss`
- Chỉ dùng `torch.softmax(logits, dim=-1)` khi inference (Task 3.7+)
- `predict_proba()` đã tích hợp eval mode + no_grad + softmax

## Task 3.4 — Class-weighted Cross-Entropy Loss

### Đang làm gì
Tính trọng số cho từng trạng thái dựa trên tần suất frame, sau đó cấu hình
`nn.CrossEntropyLoss` với các trọng số này.

### Tại sao cần trọng số
5 trạng thái cực kỳ mất cân bằng ở cấp frame (từ Task 3.1):
- Diastole: 39% frame có nhãn  → phổ biến nhất → trọng số THẤP
- Murmur  :  3% frame có nhãn  → hiếm nhất    → trọng số CAO
Nếu không có trọng số, mô hình học cách dự đoán Diastole liên tục
và bỏ qua Murmur — đây chính xác là điều ta không muốn.

### Cách tính (từ bài báo)
    weight[i] = total_labelled_frames / (n_classes × count[i])
Tần suất nghịch đảo: trạng thái càng hiếm → trọng số càng cao.
Chỉ đếm frame có nhãn hợp lệ (≥ 0), bỏ qua Unannotated (-1).

### ignore_index=-1
Hai loại frame bị bỏ qua khi tính loss:
- Unannotated (từ TSV gốc): không có ground truth
- Padding (từ collate_fn): không có dữ liệu thực

### Kết quả mong muốn
- Trọng số Murmur cao nhất, Diastole thấp nhất
- Loss tính được trên batch thực không lỗi

In [42]:
## Tính class weights
import torch
import numpy as np
from pathlib import Path

# Đếm frame theo trạng thái trên toàn bộ tập train
# (dùng stats đã có từ Task 3.1)
state_names  = ['S1', 'Systole', 'S2', 'Diastole', 'Murmur']
frame_counts = [
    stats['frame_counts'][0],  # S1
    stats['frame_counts'][1],  # Systole
    stats['frame_counts'][2],  # S2
    stats['frame_counts'][3],  # Diastole
    stats['frame_counts'][4],  # Murmur
]

n_classes       = 5
total_labelled  = sum(frame_counts)

# Công thức: weight[i] = total / (n_classes × count[i])
weights = [total_labelled / (n_classes * c) for c in frame_counts]

# Chuẩn hoá: trọng số nhỏ nhất = 1.0 (dễ đọc hơn)
min_w   = min(weights)
weights = [w / min_w for w in weights]

weights_tensor = torch.FloatTensor(weights)

print("Frame counts và class weights:\n")
print(f"{'State':>12}  {'Frames':>10}  {'% labelled':>10}  {'Weight':>8}")
print("-" * 50)
for name, count, w in zip(state_names, frame_counts, weights):
    pct = 100 * count / total_labelled
    print(f"{name:>12}  {count:>10,}  {pct:>9.1f}%  {w:>8.3f}")

print(f"\nweights_tensor: {weights_tensor}")

Frame counts và class weights:

       State      Frames  % labelled    Weight
--------------------------------------------------
          S1     385,582       20.3%     1.922
     Systole     378,660       19.9%     1.957
          S2     337,989       17.8%     2.193
    Diastole     741,055       39.0%     1.000
      Murmur      55,591        2.9%    13.330

weights_tensor: tensor([ 1.9219,  1.9570,  2.1925,  1.0000, 13.3305])


In [43]:
##Tạo loss function và kiểm tra
import torch.nn as nn

criterion = nn.CrossEntropyLoss(
    weight=weights_tensor,
    ignore_index=-1,
)
print(f"Loss function: {criterion}")

# Kiểm tra nhanh trên batch thực từ DataLoader
all_ids    = recordings_df['recording_id'].tolist()
tmp_loader = create_dataloader(all_ids, SPEC_DIR, LABEL_DIR,
                               batch_size=4, shuffle=False)
batch = next(iter(tmp_loader))

model.eval()
with torch.no_grad():
    logits = model(batch['features'], batch['lengths'])  # (B, T_max, 5)

# CrossEntropyLoss nhận (B, C, T) — cần permute
# logits: (B, T, 5) → (B, 5, T)
loss = criterion(logits.permute(0, 2, 1), batch['labels'])
print(f"\nLoss trên batch đầu tiên: {loss.item():.4f}")
print(f"(Kỳ vọng gần log(5) ≈ 1.609 với mô hình chưa train)")
print("✅ Loss tính được không lỗi")

Loss function: CrossEntropyLoss()

Loss trên batch đầu tiên: 1.5870
(Kỳ vọng gần log(5) ≈ 1.609 với mô hình chưa train)
✅ Loss tính được không lỗi


#### Task 3.4 — Kết quả

##### Class weights (tần suất nghịch đảo, chuẩn hoá về Diastole=1.0)

| Trạng thái | Frames | % labelled | Weight |
|------------|--------|-----------|--------|
| S1         | 385,582 | 20.3% | 1.922 |
| Systole    | 378,660 | 19.9% | 1.957 |
| S2         | 337,989 | 17.8% | 2.193 |
| Diastole   | 741,055 | 39.0% | 1.000 |
| Murmur     |  55,591 |  2.9% | 13.331 |

##### Kiểm chứng
- Loss trên batch chưa train: **1.587** ≈ log(5) = 1.609 ✅
- `criterion.weight` đúng tensor weights ✅
- `ignore_index=-1` bỏ qua Unannotated + padding ✅

##### Lưu ý quan trọng
- Murmur weight **13×** so với Diastole — buộc mô hình chú ý đến
  các frame murmur hiếm thay vì tối ưu theo đa số
- Khi tính loss: logits cần **permute** từ `(B, T, 5)` → `(B, 5, T)`
  vì `CrossEntropyLoss` nhận class dimension ở vị trí thứ 2

## Task 3.5 — Chia 5-Fold Cross-Validation phân tầng

### Đang làm gì
Chia 942 bệnh nhân thành 5 fold, đảm bảo tỷ lệ lớp murmur
(Present/Unknown/Absent) đồng đều qua các fold, và tất cả bản ghi
của cùng bệnh nhân nằm trong cùng fold.

### Tại sao phân tầng theo bệnh nhân, không theo bản ghi
Nếu chia theo bản ghi, mô hình có thể thấy bản ghi AV của bệnh nhân X
khi train, rồi được test trên bản ghi MV của chính bệnh nhân X đó —
đây là data leakage. Phải chia ở cấp bệnh nhân.

### Tại sao cần phân tầng (stratified)
Present chỉ chiếm 19%, Unknown 7.2% — nếu chia ngẫu nhiên, một fold
có thể không có đủ bệnh nhân Present để train/validate đúng cách.
Stratified đảm bảo mỗi fold có ~19% Present, ~7% Unknown, ~74% Absent.

### Kết quả mong muốn
- 5 fold, mỗi fold ~188 bệnh nhân validation, ~754 bệnh nhân train
- Tỷ lệ lớp murmur tương đồng qua các fold
- Không bệnh nhân nào xuất hiện trong cả train lẫn val cùng fold
- Lưu vào `data/metadata/cv_splits.json` để tái tạo được

In [46]:
import json
from sklearn.model_selection import StratifiedKFold

# Chuẩn bị dữ liệu cấp bệnh nhân
patient_ids = patients_df['patient_id'].tolist()
labels_pat  = patients_df['murmur'].tolist()   # Present / Unknown / Absent

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_splits = {}

for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(patient_ids, labels_pat)):

    train_patients = [patient_ids[i] for i in train_idx]
    val_patients   = [patient_ids[i] for i in val_idx]

    # Mở rộng từ cấp bệnh nhân sang cấp bản ghi
    train_recordings = recordings_df[
        recordings_df['patient_id'].isin(train_patients)
    ]['recording_id'].tolist()

    val_recordings = recordings_df[
        recordings_df['patient_id'].isin(val_patients)
    ]['recording_id'].tolist()

    cv_splits[f'fold_{fold_idx}'] = {
        'train_patients':   [int(p) for p in train_patients],
        'val_patients':     [int(p) for p in val_patients],
        'train_recordings': train_recordings,
        'val_recordings':   val_recordings,
    }

# Lưu ra file
splits_path = META_DIR / 'cv_splits.json'
with open(splits_path, 'w') as f:
    json.dump(cv_splits, f, indent=2)

print(f"Đã lưu: {splits_path}")

Đã lưu: D:\1 Onedrive\1 Lectures\3.2 DAV\papers\heart-murmur-detection\data\metadata\cv_splits.json


In [47]:
print("=" * 55)
print(f"{'Fold':<6} {'Train pat':>10} {'Val pat':>8} "
      f"{'Train rec':>10} {'Val rec':>8}")
print("-" * 55)

for fold_name, fold_data in cv_splits.items():
    print(f"{fold_name:<6} "
          f"{len(fold_data['train_patients']):>10} "
          f"{len(fold_data['val_patients']):>8} "
          f"{len(fold_data['train_recordings']):>10} "
          f"{len(fold_data['val_recordings']):>8}")

# Kiểm tra phân bố lớp mỗi fold
print("\nPhân bố lớp murmur mỗi fold (val set):")
print(f"{'Fold':<6} {'Present':>9} {'Unknown':>9} {'Absent':>9}")
print("-" * 40)

pat_murmur = dict(zip(patients_df['patient_id'], patients_df['murmur']))

for fold_name, fold_data in cv_splits.items():
    val_labels = [pat_murmur[p] for p in fold_data['val_patients']]
    n_present  = val_labels.count('Present')
    n_unknown  = val_labels.count('Unknown')
    n_absent   = val_labels.count('Absent')
    print(f"{fold_name:<6} {n_present:>8} ({100*n_present/len(val_labels):4.1f}%)"
          f"  {n_unknown:>5} ({100*n_unknown/len(val_labels):4.1f}%)"
          f"  {n_absent:>5} ({100*n_absent/len(val_labels):4.1f}%)")

# Kiểm tra không data leakage
print("\nKiểm tra data leakage (không bệnh nhân nào trong cả train lẫn val):")
all_clean = True
for fold_name, fold_data in cv_splits.items():
    overlap = set(fold_data['train_patients']) & set(fold_data['val_patients'])
    if overlap:
        print(f"  ❌ {fold_name}: {len(overlap)} bệnh nhân bị trùng!")
        all_clean = False
if all_clean:
    print("  ✅ Không có data leakage")

# Kiểm tra mỗi bệnh nhân xuất hiện đúng 1 lần trong val
all_val_patients = []
for fold_data in cv_splits.values():
    all_val_patients.extend(fold_data['val_patients'])

print(f"\nTổng bệnh nhân qua 5 fold val: {len(all_val_patients)}")
print(f"Unique bệnh nhân             : {len(set(all_val_patients))}")
print(f"Khớp tổng dataset            : {len(set(all_val_patients)) == len(patient_ids)}")

Fold    Train pat  Val pat  Train rec  Val rec
-------------------------------------------------------
fold_0        753      189       2553      610
fold_1        753      189       2516      647
fold_2        754      188       2519      644
fold_3        754      188       2535      628
fold_4        754      188       2529      634

Phân bố lớp murmur mỗi fold (val set):
Fold     Present   Unknown    Absent
----------------------------------------
fold_0       36 (19.0%)     14 ( 7.4%)    139 (73.5%)
fold_1       36 (19.0%)     14 ( 7.4%)    139 (73.5%)
fold_2       36 (19.1%)     13 ( 6.9%)    139 (73.9%)
fold_3       36 (19.1%)     13 ( 6.9%)    139 (73.9%)
fold_4       35 (18.6%)     14 ( 7.4%)    139 (73.9%)

Kiểm tra data leakage (không bệnh nhân nào trong cả train lẫn val):
  ✅ Không có data leakage

Tổng bệnh nhân qua 5 fold val: 942
Unique bệnh nhân             : 942
Khớp tổng dataset            : True


#### ✅ Task 3.5 — Kết quả

##### Artifacts tạo ra
- `data/metadata/cv_splits.json` — 5 fold splits có thể tái tạo (seed=42)

##### Cấu trúc mỗi fold

| Fold | Train patients | Val patients | Train recordings | Val recordings |
|------|:---:|:---:|:---:|:---:|
| fold_0 | 753 | 189 | 2553 | 610 |
| fold_1 | 753 | 189 | 2516 | 647 |
| fold_2 | 754 | 188 | 2519 | 644 |
| fold_3 | 754 | 188 | 2535 | 628 |
| fold_4 | 754 | 188 | 2529 | 634 |

##### Phân bố lớp (val set mỗi fold)
Present ~19%, Unknown ~7%, Absent ~74% — đồng đều qua tất cả fold ✅

##### Kiểm chứng
- Data leakage: không có ✅
- Tổng val = 942 unique bệnh nhân, mỗi người đúng 1 fold ✅

## Task 3.6 — Huấn luyện RNN (5-fold Cross-Validation)

### Đang làm gì
Huấn luyện MurmurRNN trên 5 fold, mỗi fold lưu model checkpoint tốt nhất
(val loss thấp nhất) và ghi log đường loss train/val theo epoch.

### Hyperparameters
- Optimiser: Adam, lr=1e-3
- Batch size: 16
- Max epochs: 100
- Early stopping: dừng nếu val loss không cải thiện sau 10 epoch
- Gradient clipping: max_norm=1.0 (ổn định GRU)

### Kết quả mong muốn
- 5 file checkpoint: `models/rnn/fold{k}_best.pt`
- 5 file loss log  : `experiments/logs/fold{k}_loss.csv`
- Val loss giảm và hội tụ qua các epoch
- Không overfitting nghiêm trọng (khoảng cách train/val hợp lý)

### Ước tính thời gian (CPU)
~20–40 phút/fold × 5 fold = 1.5–3 giờ tổng

In [49]:
import torch
import torch.nn as nn
import csv
from pathlib import Path
from src.models.rnn import build_model
from src.data.loader import create_dataloader

# ── Config ────────────────────────────────────────────────────────────
CONFIG = {
    'lr':            1e-3,
    'batch_size':    16,
    'max_epochs':    100,
    'patience':      10,     # early stopping
    'grad_clip':     1.0,
    'seed':          42,
    'num_workers':   0,      # Windows: phải để 0
}

# ── Paths ─────────────────────────────────────────────────────────────
MODELS_DIR = project_root / 'models' / 'rnn'
LOGS_DIR   = project_root / 'experiments' / 'logs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:>12}: {v}")
print(f"\nModels dir: {MODELS_DIR}")
print(f"Logs dir  : {LOGS_DIR}")

Config:
            lr: 0.001
    batch_size: 16
    max_epochs: 100
      patience: 10
     grad_clip: 1.0
          seed: 42
   num_workers: 0

Models dir: D:\1 Onedrive\1 Lectures\3.2 DAV\papers\heart-murmur-detection\models\rnn
Logs dir  : D:\1 Onedrive\1 Lectures\3.2 DAV\papers\heart-murmur-detection\experiments\logs


In [51]:
# Đọc T thực từ file .npy —> lưu vào file recordings
n_frames_list = []
for rec_id in tqdm(recordings_df['recording_id'], desc="Reading T"):
    spec = np.load(SPEC_DIR / f"{rec_id}.npy", mmap_mode='r')
    n_frames_list.append(spec.shape[1])

recordings_df['n_frames'] = n_frames_list

# Lưu lại CSV
recordings_df.to_csv(META_DIR / "recordings.csv", index=False)
print(f"Đã lưu recordings.csv với cột n_frames")
print(recordings_df[['recording_id', 'duration_seconds', 'n_frames']].head(5))

Reading T: 100%|██████████| 3163/3163 [00:14<00:00, 225.37it/s]


Đã lưu recordings.csv với cột n_frames
  recording_id  duration_seconds  n_frames
0      2530_AV            23.600      1181
1      2530_PV            13.056       654
2      2530_TV            14.992       751
3      2530_MV            26.688      1336
4      9979_AV            18.160       909


In [52]:
lengths_dict = dict(zip(recordings_df['recording_id'],
                        recordings_df['n_frames']))
print(f"lengths_dict: {len(lengths_dict)} recordings — from CSV, no disk I/O")

lengths_dict: 3163 recordings — from CSV, no disk I/O


In [66]:
### mỗi lần restart kernel phải chạy cell reload này trước cell huấn luyện
import importlib
import src.data.loader
importlib.reload(src.data.loader)
from src.data.loader import load_dataset_to_ram, create_dataloader

print(dir(src.data.loader))  # kiểm tra load_dataset_to_ram đã có chưa

['DataLoader', 'Dataset', 'PCGDataset', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'create_dataloader', 'load_dataset_to_ram', 'np', 'pcg_collate_fn', 'torch', 'tqdm']


In [65]:
# Load toàn bộ dataset vào RAM — chạy 1 lần duy nhất
from src.data.loader import load_dataset_to_ram, create_dataloader

spec_cache, label_cache, lengths_dict = load_dataset_to_ram(
    recordings_df['recording_id'].tolist(),
    SPEC_DIR,
    LABEL_DIR
)

Loading dataset to RAM: 100%|██████████| 3163/3163 [00:28<00:00, 109.11it/s]


RAM used: ~1191 MB — 3163 recordings cached


In [67]:
import time
from tqdm import tqdm
def run_epoch(model, loader, criterion, optimiser=None, is_train=True,
              fold_idx=0, epoch=0):
    model.train() if is_train else model.eval()
    total_loss = 0.0
    n_batches  = 0
    mode       = "train" if is_train else "val"

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, desc=f"  Fold {fold_idx} Ep {epoch:>3} {mode}",
                    leave=False, ncols=80)
        for batch in pbar:
            features = batch['features']
            labels   = batch['labels']
            lengths  = batch['lengths']

            logits = model(features, lengths)
            loss   = criterion(logits.permute(0, 2, 1), labels)

            if is_train:
                optimiser.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),
                                               CONFIG['grad_clip'])
                optimiser.step()

            total_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / n_batches if n_batches > 0 else 0.0


# ── 5-fold training loop ───────────────────────────────────────────────
torch.manual_seed(CONFIG['seed'])

for fold_idx in range(5):
    fold_name = f'fold_{fold_idx}'
    print(f"\n{'='*55}")
    print(f"  {fold_name.upper()}  ({fold_idx+1}/5)")
    print(f"{'='*55}")

    # DataLoaders
    train_ids = cv_splits[fold_name]['train_recordings']
    val_ids   = cv_splits[fold_name]['val_recordings']

    train_loader = create_dataloader(
        train_ids, SPEC_DIR, LABEL_DIR,
        batch_size=CONFIG['batch_size'], shuffle=True,
        num_workers=CONFIG['num_workers'],
        spec_cache=spec_cache,
        label_cache=label_cache,
        lengths_dict=lengths_dict
    )
    val_loader = create_dataloader(
        val_ids, SPEC_DIR, LABEL_DIR,
        batch_size=CONFIG['batch_size'], shuffle=False,
        num_workers=CONFIG['num_workers'],
        spec_cache=spec_cache,
        label_cache=label_cache,
        lengths_dict=lengths_dict
    )
    print(f"  Train: {len(train_ids)} recordings, "
          f"{len(train_loader)} batches")
    print(f"  Val  : {len(val_ids)} recordings, "
          f"{len(val_loader)} batches")

    # Model + optimiser (fresh mỗi fold)
    torch.manual_seed(CONFIG['seed'])
    model     = build_model(seed=CONFIG['seed'])
    optimiser = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.CrossEntropyLoss(
        weight=weights_tensor,
        ignore_index=-1
    )

    # Training state
    best_val_loss  = float('inf')
    patience_count = 0
    log_rows       = []
    fold_start     = time.time()

    for epoch in range(1, CONFIG['max_epochs'] + 1):
        ep_start    = time.time()
        train_loss = run_epoch(model, train_loader, criterion,
                               optimiser, is_train=True,
                               fold_idx=fold_idx, epoch=epoch)
        val_loss   = run_epoch(model, val_loader, criterion,
                               is_train=False,
                               fold_idx=fold_idx, epoch=epoch)
        ep_secs     = time.time() - ep_start

        log_rows.append({
            'epoch': epoch,
            'train_loss': round(train_loss, 6),
            'val_loss':   round(val_loss,   6),
        })

        # In progress mỗi 5 epoch
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3}  "
                  f"train={train_loss:.4f}  "
                  f"val={val_loss:.4f}  "
                  f"({ep_secs:.0f}s)")

        # Lưu checkpoint nếu val loss tốt hơn
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            torch.save({
                'epoch':      epoch,
                'model_state_dict': model.state_dict(),
                'val_loss':   val_loss,
                'config':     CONFIG,
            }, MODELS_DIR / f'{fold_name}_best.pt')
        else:
            patience_count += 1

        # Early stopping
        if patience_count >= CONFIG['patience']:
            print(f"  Early stopping tại epoch {epoch} "
                  f"(no improvement for {CONFIG['patience']} epochs)")
            break

    # Lưu loss log
    log_path = LOGS_DIR / f'{fold_name}_loss.csv'
    with open(log_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss'])
        writer.writeheader()
        writer.writerows(log_rows)

    fold_mins = (time.time() - fold_start) / 60
    print(f"\n  ✅ {fold_name} done — "
          f"best val loss: {best_val_loss:.4f} "
          f"({fold_mins:.1f} min)")
    print(f"  Checkpoint: {MODELS_DIR/f'{fold_name}_best.pt'}")
    print(f"  Log       : {log_path}")

print(f"\n{'='*55}")
print("✅ Huấn luyện hoàn tất — 5/5 fold")


  FOLD_0  (1/5)
  Train: 2553 recordings, 160 batches
  Val  : 610 recordings, 39 batches


KeyboardInterrupt: 

In [69]:
import time

# Đo thời gian từng phần trong 1 batch
batch = next(iter(train_loader))
features = batch['features']
lengths  = batch['lengths']
labels   = batch['labels']

print(f"Batch features shape: {features.shape}")
print(f"T_max trong batch này: {lengths[0]}")

# Đo forward pass
t0 = time.time()
model.train()
logits = model(features, lengths)
print(f"Forward pass: {time.time()-t0:.2f}s")

# Đo backward pass
t0 = time.time()
loss = criterion(logits.permute(0, 2, 1), labels)
loss.backward()
print(f"Backward pass: {time.time()-t0:.2f}s")

Batch features shape: torch.Size([16, 1971, 41])
T_max trong batch này: 1971
Forward pass: 6.10s
Backward pass: 64.09s
